## Face Mask Detection — FPGA Optimized Model (Consistent Accuracy)

In [21]:
import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras

# ── Fix all random seeds for consistency ──
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print("✅ Seeds fixed")

✅ Seeds fixed


## Data Loading

In [22]:
class FaceMaskDatasetLoader:
    def __init__(self):
        self.files = []
    def load_files(self, paths):
        for path in paths:
            if os.path.exists(path):
                # Sort for consistent order every run
                self.files.extend(sorted([os.path.join(path, f) for f in os.listdir(path)]))
            else:
                print(f"Directory not found: {path}")
        return self.files

with_mask_paths = [
    '/kaggle/input/face-mask-dataset/data/with_mask',
    '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Train/WithMask',
    '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Test/WithMask',
    '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Validation/WithMask'
]
without_mask_paths = [
    '/kaggle/input/face-mask-dataset/data/without_mask',
    '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Train/WithoutMask',
    '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Test/WithoutMask',
    '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Validation/WithoutMask'
]

loader = FaceMaskDatasetLoader()
with_mask_files = loader.load_files(with_mask_paths)
loader = FaceMaskDatasetLoader()
without_mask_files = loader.load_files(without_mask_paths)
print(f"With mask: {len(with_mask_files)} | Without mask: {len(without_mask_files)}")

With mask: 9608 | Without mask: 9737


## Data Preprocessing

In [23]:
class DataProcessor:
    def __init__(self, image_size=(128, 128)):
        self.image_size = image_size
    def process_images(self, with_mask_files, without_mask_files):
        data, labels = [], []
        all_files = [(f, 1) for f in with_mask_files] + [(f, 0) for f in without_mask_files]
        for file_path, label in all_files:
            try:
                image = Image.open(file_path).resize(self.image_size).convert('RGB')
                data.append(np.array(image))
                labels.append(label)
            except Exception as e:
                print(f"Error: {file_path}: {e}")
        data   = np.array(data)   / 255.0
        labels = np.array(labels)
        return data, labels

# 64x64 for FPGA
processor = DataProcessor(image_size=(64, 64))
data, labels = processor.process_images(with_mask_files, without_mask_files)
print("Data shape:", data.shape)

X_train, X_test, Y_train, Y_test = train_test_split(
    data, labels, test_size=0.2, shuffle=True, random_state=11
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Data shape: (19345, 64, 64, 3)
Train: (15476, 64, 64, 3) | Test: (3869, 64, 64, 3)


## Train Once — Save Best Model
If `best_fpga_model.keras` already exists, skip training and load it directly.

In [24]:
# MODEL_PATH = '/kaggle/working/best_fpga_model.keras'

# if os.path.exists(MODEL_PATH):
#     # ── LOAD existing best model ──
#     print("✅ Found saved model — loading directly (no retraining)")
#     model = tf.keras.models.load_model(MODEL_PATH)
#     print("✅ Model loaded — accuracy will be identical to last save")

# else:
#     # ── TRAIN for the first time ──
#     print("No saved model found — training now...")

#     # Class weights
#     class_weights = compute_class_weight(
#         class_weight='balanced',
#         classes=np.unique(Y_train),
#         y=Y_train
#     )
#     class_weight_dict = dict(enumerate(class_weights))
#     print("Class weights:", class_weight_dict)

#     # ── FPGA OPTIMIZED ARCHITECTURE ──
#     model = keras.Sequential()
#     model.add(keras.layers.Conv2D(16, kernel_size=(3,3), activation='relu', input_shape=(64,64,3)))
#     model.add(keras.layers.MaxPooling2D(pool_size=(2,2)))
#     model.add(keras.layers.Conv2D(32, kernel_size=(3,3), activation='relu'))
#     model.add(keras.layers.MaxPooling2D(pool_size=(2,2)))
#     model.add(keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu'))
#     model.add(keras.layers.MaxPooling2D(pool_size=(2,2)))
#     model.add(keras.layers.Flatten())
#     model.add(keras.layers.Dense(64, activation='relu'))
#     model.add(keras.layers.Dropout(0.5))
#     model.add(keras.layers.Dense(32, activation='relu'))
#     model.add(keras.layers.Dropout(0.5))
#     model.add(keras.layers.Dense(2))  # no sigmoid — logits

#     model.compile(
#         optimizer='adam',
#         loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#         metrics=['accuracy']
#     )

#     from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

#     checkpoint = ModelCheckpoint(
#         MODEL_PATH,
#         monitor='val_accuracy',
#         save_best_only=True,
#         verbose=1
#     )
#     early_stop = EarlyStopping(
#         monitor='val_accuracy',
#         patience=3,
#         restore_best_weights=True,
#         verbose=1
#     )

#     history = model.fit(
#         X_train, Y_train,
#         validation_split=0.1,
#         epochs=20,
#         class_weight=class_weight_dict,
#         callbacks=[checkpoint, early_stop]
#     )

#     print("\n✅ Training complete — best model saved to:", MODEL_PATH)

import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"

import tensorflow as tf
tf.config.optimizer.set_jit(False)

MODEL_PATH = '/kaggle/working/best_fpga_model.keras'

if os.path.exists(MODEL_PATH):
    print("✅ Found saved model — loading directly (no retraining)")
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Model loaded — accuracy will be identical to last save")

else:
    print("No saved model found — training now...")

    # Class weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(Y_train),
        y=Y_train
    )
    class_weight_dict = dict(enumerate(class_weights))
    print("Class weights:", class_weight_dict)

    # ── FPGA OPTIMIZED (NO MAXPOOL) ──
    model = keras.Sequential()

    model.add(keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(64,64,3)))
    model.add(keras.layers.Conv2D(16, (3,3), strides=2, padding='same', activation='relu'))

    model.add(keras.layers.Conv2D(32, (3,3), activation='relu'))
    model.add(keras.layers.Conv2D(32, (3,3), strides=2, padding='same', activation='relu'))

    model.add(keras.layers.Conv2D(64, (3,3), activation='relu'))
    model.add(keras.layers.Conv2D(64, (3,3), strides=2, padding='same', activation='relu'))

    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(64, activation='relu'))
    model.add(keras.layers.Dropout(0.5))
    model.add(keras.layers.Dense(32, activation='relu'))
    model.add(keras.layers.Dropout(0.5))
    model.add(keras.layers.Dense(2))

    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

    checkpoint = ModelCheckpoint(
        MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    )

    history = model.fit(
        X_train, Y_train,
        validation_split=0.1,
        epochs=20,
        class_weight=class_weight_dict,
        callbacks=[checkpoint, early_stop]
    )

    print("\n✅ Training complete — best model saved to:", MODEL_PATH)

✅ Found saved model — loading directly (no retraining)
✅ Model loaded — accuracy will be identical to last save


## Evaluation — Consistent Every Run

In [25]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"\nTest Accuracy = {accuracy:.4f}")

121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9639 - loss: 0.1184

Test Accuracy = 0.9630


In [26]:
train_acc = history.history['accuracy'][-1]
val_acc   = history.history['val_accuracy'][-1]

print("\n📈 TRAINING METRICS")
print("==========================")
print(f"Final Training Accuracy : {train_acc:.4f}")
print(f"Final Validation Accuracy (Test) : {val_acc:.4f}")


📈 TRAINING METRICS
Final Training Accuracy : 0.9759
Final Validation Accuracy (Test) : 0.9716


## Dataset Summary

In [27]:
from tqdm import tqdm

def evaluate_dataset(model, image_size=(64, 64)):
    dataset_path = "/kaggle/input/face-mask-dataset/data"
    class_folders = {"with_mask": 1, "without_mask": 0}
    images, labels_list = [], []

    for folder, label in class_folders.items():
        folder_path = os.path.join(dataset_path, folder)
        for img_name in tqdm(sorted(os.listdir(folder_path)), desc=f"Loading {folder}"):
            img_path = os.path.join(folder_path, img_name)
            try:
                image = Image.open(img_path).resize(image_size).convert('RGB')
                images.append(np.array(image) / 255.0)
                labels_list.append(label)
            except: pass

    images     = np.array(images)
    labels_arr = np.array(labels_list)
    preds      = np.argmax(model.predict(images, batch_size=32, verbose=1), axis=1)

    total       = len(labels_arr)
    correct     = np.sum(preds == labels_arr)
    wm_total    = np.sum(labels_arr == 1)
    wom_total   = np.sum(labels_arr == 0)
    wm_correct  = np.sum((labels_arr == 1) & (preds == 1))
    wom_correct = np.sum((labels_arr == 0) & (preds == 0))

    print("\n==========================")
    print("📊 DATASET SUMMARY")
    print("==========================")
    print(f"Total Images        : {total}")
    print(f"Correct Predictions : {correct}")
    print(f"Wrong Predictions   : {total - correct}")
    print(f"Overall Accuracy    : {correct/total:.4f}")
    print(f"\n😷 WITH MASK:")
    print(f"  Accuracy : {wm_correct/wm_total:.4f}")
    print(f"\n🚫 WITHOUT MASK:")
    print(f"  Accuracy : {wom_correct/wom_total:.4f}")
    print(f"\nBias Gap            : {abs(wm_correct/wm_total - wom_correct/wom_total):.4f}")
    print("==========================")

evaluate_dataset(model)

Loading without_mask: 100%|██████████| 3828/3828 [00:06<00:00, 560.26it/s]


237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

📊 DATASET SUMMARY
Total Images        : 7553
Correct Predictions : 7283
Wrong Predictions   : 270
Overall Accuracy    : 0.9643

😷 WITH MASK:
  Accuracy : 0.9715

🚫 WITHOUT MASK:
  Accuracy : 0.9572

Bias Gap            : 0.0144


## Export to ONNX for Tensil/FPGA

In [28]:
import tf2onnx, onnx

input_signature = [tf.TensorSpec(shape=(1, 64, 64, 3), dtype=tf.float32)]
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature)
onnx.save(onnx_model, "best_fpga_model.onnx")
print("✅ ONNX export done — ready for Tensil compilation!")

ModuleNotFoundError: No module named 'tf2onnx'